# 02 - 3D Galactic dust maps (E(B-V) as a function of distance)

Inspection of the 3D dust maps stored in `maps/DustMaps3D`. These are the maps used by
`rubin_sim.maf.maps.DustMap3D` (contributors: W. Clarkson, A. Mazzi); the `README` says:

* each FITS file holds, for every HEALPix pixel, an array of **distances** and the **E(B-V)** reached at each of them;
* the HEALPix grid is defined on **RA/Dec** pixel centres, in **RING** order;
* at the largest distances the maps converge towards the 2D SFD maps (see `01_DustMaps.ipynb`), although SFD
  contains much larger extinction in some parts of the plane;
* the files differ by the combination of underlying 3D maps (Lallement+2019, Marshall+2006, Green+2019, Drimmel+2003,
  Planck ...): `defaults`, `bridge`, `noL19`, and the plain `merged` map.

**Layout assumed here** (same as the `rubin_sim` reader `ebv_3d_hp`; checked in section 2):

| HDU | content |
|-----|---------|
| 0 | header (`NSIDE`, `NESTED`, `NBINS`, ...) + array of HEALPix ids |
| 1 | `dists`: distances [pc], shape `(npix, nbins)` |
| 2 | `ebvs`: E(B-V) [mag] at those distances, shape `(npix, nbins)` |

The files are large (0.4 - 1.5 GB): they are opened **memory-mapped** and processed in blocks of pixels.

**Contents**
1. Inventory and headers
2. Loading and sanity checks
3. Lines of sight: E(B-V) versus distance
4. Full-sky maps at fixed distances
5. Far end of the 3D map versus the 2D SFD map
6. Distance reached at a given apparent-minus-absolute magnitude (`m - M0`)
7. Comparison of the map variants
8. Effect of the HEALPix resolution (NSIDE 64 vs 128)

In [ ]:
from pathlib import Path
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import healpy as hp
from astropy.io import fits

%matplotlib inline
plt.rcParams.update({"figure.dpi": 110, "font.size": 11})

# The notebook lives in notebooks/09_RubinMaps/ and the maps were copied to ./maps
CANDIDATES = [
    Path("maps/DustMaps3D"),
    Path.home() / "DATA/OpSim/maps/DustMaps3D",
]
DATA_DIR = next(p for p in CANDIDATES if p.is_dir())
DUST2D_DIR = DATA_DIR.parent / "DustMaps"  # 2D SFD maps (see 01_DustMaps.ipynb)

FILES = {
    "defaults": "merged_ebv3d_nside64_defaults.fits",
    "bridge": "merged_ebv3d_nside64_bridge.fits",
    "noL19": "merged_ebv3d_nside64_noL19.fits",
    "merged": "merged_ebv3d_nside64.fits",
    "defaults128": "merged_ebv3d_nside128_defaults.fits",
}
FILES = {name: f for name, f in FILES.items() if (DATA_DIR / f).exists()}
REF = "defaults" if "defaults" in FILES else next(iter(FILES))  # reference map for the single-map plots

# Set to False to skip the (slow, 1.5 GB) NSIDE=128 file in section 8
RUN_NSIDE128 = True

print(f"Data directory: {DATA_DIR.resolve()}")
print(f"Files found   : {list(FILES)}")

## 1. Inventory and headers

In [ ]:
KEYS = [
    "NSIDE",
    "NESTED",
    "NBINS",
    "NL",
    "NB",
    "FRACPIX",
    "RV",
    "MAPVERS",
    "PLANCKOK",
    "DMAXL19",
    "BRIDGL19",
    "BRIDGWID",
    "HPMIN",
    "HPMAX",
]

table = []
for name, fname in FILES.items():
    path = DATA_DIR / fname
    hdr = fits.getheader(path, 0)
    table.append(
        {"name": name, "size [MiB]": round(path.stat().st_size / 2**20, 1), **{k: hdr.get(k) for k in KEYS}}
    )
header_table = pd.DataFrame(table).set_index("name")
header_table

In [ ]:
# HDU structure of the reference file (memory-mapped: only headers are read)
with fits.open(DATA_DIR / FILES[REF], memmap=True) as hdul:
    hdul.info()

## 2. Loading and sanity checks

`DustCube` gives lazy access to one file. `rows` maps each RING pixel index to its row in the arrays
(identity for RING files; a re-ordering if a file were stored in NESTED order).

In [ ]:
class DustCube:
    """Lazy (memory-mapped) access to one merged_ebv3d_*.fits file."""

    def __init__(self, path):
        self.path = Path(path)
        self.hdul = fits.open(self.path, memmap=True)
        self.header = self.hdul[0].header
        self.nside = int(self.header["NSIDE"])
        self.nested = bool(self.header["NESTED"])
        self.npix = hp.nside2npix(self.nside)
        self.hpids = np.asarray(self.hdul[0].data)
        self.dists = self.hdul[1].data  # (npix, nbins) distances [pc]
        self.ebvs = self.hdul[2].data  # (npix, nbins) E(B-V) [mag]
        assert self.dists.shape == self.ebvs.shape and self.dists.shape[0] == self.npix
        ring = np.arange(self.npix)
        self.rows = hp.ring2nest(self.nside, ring) if self.nested else ring

    @property
    def nbins(self):
        return self.dists.shape[1]

    def close(self):
        self.hdul.close()


cubes = {name: DustCube(DATA_DIR / f) for name, f in FILES.items()}

for name, c in cubes.items():
    print(
        f"{name:12s} NSIDE={c.nside:4d}  npix={c.npix:7d}  nbins={c.nbins:4d}  "
        f"dtype={c.dists.dtype}/{c.ebvs.dtype}  hpids==arange: {np.array_equal(c.hpids, np.arange(c.npix))}"
    )

In [ ]:
def iter_chunks(cube, chunk=4096):
    """Yield (RING-pixel slice, distances, E(B-V)) for blocks of pixels, as float64 arrays."""
    for s in range(0, cube.npix, chunk):
        sl = slice(s, min(s + chunk, cube.npix))
        idx = cube.rows[sl] if cube.nested else sl
        yield sl, np.asarray(cube.dists[idx], dtype=float), np.asarray(cube.ebvs[idx], dtype=float)


def at_distance(cube, d_pc):
    """E(B-V) [mag] and actual grid distance [pc] at the distance bin nearest to d_pc (RING-ordered maps)."""
    ebv_map = np.full(cube.npix, np.nan)
    dist_map = np.full(cube.npix, np.nan)
    for sl, d, e in iter_chunks(cube):
        gap = np.where(np.isfinite(d), np.abs(d - d_pc), np.inf)
        k = gap.argmin(axis=1)
        r = np.arange(d.shape[0])
        ebv_map[sl], dist_map[sl] = e[r, k], d[r, k]
    return ebv_map, dist_map


def farthest(cube):
    """E(B-V) [mag] and distance [pc] in the last distance bin of every pixel (RING-ordered maps)."""
    rows = cube.rows
    return (np.asarray(cube.ebvs[:, -1], dtype=float)[rows], np.asarray(cube.dists[:, -1], dtype=float)[rows])


def distance_at_mag(cube, d_mag, r_x, tol=0.5):
    """Distance [pc] where m - M0 = 5 log10(d / 10 pc) + r_x * E(B-V)(d) is closest to d_mag (RING-ordered map).

    NaN where even the closest distance bin is more than `tol` mag away from d_mag.
    """
    out = np.full(cube.npix, np.nan)
    for sl, d, e in iter_chunks(cube):
        with np.errstate(divide="ignore", invalid="ignore"):
            mm = 5.0 * np.log10(d / 10.0) + r_x * e
        gap = np.where(np.isfinite(mm), np.abs(mm - d_mag), np.inf)
        k = gap.argmin(axis=1)
        r = np.arange(d.shape[0])
        out[sl] = np.where(gap[r, k] <= tol, d[r, k], np.nan)
    return out


def pixel_lb(nside):
    """Galactic (l, b) in degrees of the pixel centres (the maps are defined on RA/Dec pixel centres)."""
    ra, dec = hp.pix2ang(nside, np.arange(hp.nside2npix(nside)), lonlat=True)
    return hp.Rotator(coord=["C", "G"])(ra, dec, lonlat=True)


def load_sfd(nside):
    """2D SFD E(B-V) HEALPix map (see 01_DustMaps.ipynb), or None if not available."""
    path = DUST2D_DIR / f"dust_nside_{nside}.npz"
    if not path.exists():
        return None
    with np.load(path) as f:
        return f["ebvMap"]

In [ ]:
# Sanity checks on a random sample of pixels of the reference map
ref = cubes[REF]
rng = np.random.default_rng(0)
rows = np.sort(rng.choice(ref.npix, size=min(4000, ref.npix), replace=False))
d_s = np.asarray(ref.dists[rows], dtype=float)
e_s = np.asarray(ref.ebvs[rows], dtype=float)

print(f"Reference map: {REF}  ({d_s.shape[0]} sampled pixels, {d_s.shape[1]} distance bins)")
print(
    f"  non-finite values      : distances {(~np.isfinite(d_s)).mean():.2%}, E(B-V) {(~np.isfinite(e_s)).mean():.2%}"
)
print(f"  first-bin distance [pc]: min {np.nanmin(d_s[:, 0]):.1f}, max {np.nanmax(d_s[:, 0]):.1f}")
print(f"  last-bin distance [pc] : min {np.nanmin(d_s[:, -1]):.1f}, max {np.nanmax(d_s[:, -1]):.1f}")
spread = np.nanmax(d_s, axis=0) - np.nanmin(d_s, axis=0)
print(f"  same distance grid for all sampled pixels: {bool(np.all(spread <= 1e-6 * np.nanmax(d_s)))}")
steps = np.nan_to_num(np.diff(e_s, axis=1), nan=0.0)
print(f"  pixels with E(B-V) non-decreasing with distance: {np.mean(np.all(steps >= -1e-9, axis=1)):.2%}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for r in range(min(8, d_s.shape[0])):
    axes[0].semilogy(d_s[r], lw=0.8)
axes[0].set(xlabel="distance bin index", ylabel="distance [pc]", title="Distance grid of a few pixels")
last = e_s[:, -1][np.isfinite(e_s[:, -1]) & (e_s[:, -1] > 0)]
axes[1].hist(last, bins=np.logspace(np.log10(last.min()), np.log10(last.max()), 100))
axes[1].set(
    xscale="log",
    yscale="log",
    xlabel="E(B-V) in the last distance bin [mag]",
    ylabel="number of pixels",
    title="Far end of the map (sampled pixels)",
)
plt.tight_layout()
plt.show()

## 3. Lines of sight: E(B-V) versus distance

Cumulative E(B-V) along a few directions, for each map variant. The dotted horizontal line is the 2D SFD value
(integrated to infinity) in the same pixel, to which the 3D maps should converge at large distance.

In [ ]:
LOS = {
    "Galactic centre (l=0, b=0)": (0, 0),
    "Baade's window (l=1, b=-4)": (1, -4),
    "Plane, l=90": (90, 0),
    "Anticentre (l=180, b=0)": (180, 0),
    "Mid-latitude (l=270, b=-30)": (270, -30),
    "North Galactic Pole": (0, 90),
}


def los_curve(cube, l, b):
    """(pixel, distances [pc], E(B-V) [mag]) of the pixel containing Galactic (l, b) [deg]."""
    ra, dec = hp.Rotator(coord=["G", "C"])(l, b, lonlat=True)
    ip = int(hp.ang2pix(cube.nside, ra, dec, lonlat=True))
    row = cube.rows[ip]
    return ip, np.asarray(cube.dists[row], dtype=float), np.asarray(cube.ebvs[row], dtype=float)


sfd_ref = load_sfd(ref.nside)
variants64 = [n for n, c in cubes.items() if c.nside == ref.nside]

fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True)
for ax, (title, (l, b)) in zip(axes.flat, LOS.items()):
    for name in variants64:
        ip, d, e = los_curve(cubes[name], l, b)
        ax.loglog(d, e, label=name)
    if sfd_ref is not None:
        ax.axhline(sfd_ref[ip], color="k", ls=":", label="SFD (2D)")
    ax.set(title=title)
    ax.grid(alpha=0.3, which="both")
for ax in axes[1]:
    ax.set_xlabel("distance [pc]")
for ax in axes[:, 0]:
    ax.set_ylabel("E(B-V) [mag]")
axes[0, 0].legend(fontsize=9)
plt.tight_layout()
plt.show()

## 4. Full-sky maps at fixed distances

For every pixel the distance bin nearest to the requested distance is used (as in `rubin_sim`'s
`get_x_at_nearest_y`); if the requested distance is beyond the grid, the last bin is used.
The title gives the median distance of the bins actually picked.

In [ ]:
DISTANCES_PC = (100, 300, 1000, 3000, 10000, 30000)

t0 = time.time()
slices = {d: at_distance(ref, d) for d in DISTANCES_PC}
print(f"{len(slices)} slices computed in {time.time() - t0:.1f} s")

fig = plt.figure(figsize=(16, 9))
for i, (d, (m, d_used)) in enumerate(slices.items(), start=1):
    m = np.where(m > 0, m, np.nan)
    good = m[np.isfinite(m)]
    vmin, vmax = np.percentile(good, [2, 99.5]) if good.size else (0.01, 1)
    hp.mollview(
        m,
        coord=["C", "G"],
        norm="log",
        min=vmin,
        max=vmax,
        unit="E(B-V) [mag]",
        title=f"d ~ {d} pc (grid: {np.nanmedian(d_used):.0f} pc)",
        sub=(2, 3, i),
        fig=fig.number,
    )
    hp.graticule()
plt.show()

## 5. Far end of the 3D map versus the 2D SFD map

E(B-V) in the last distance bin of each pixel compared with the 2D SFD map. The maps should converge, except in
the Galactic plane where SFD reaches much larger values.

In [ ]:
ebv_far, dist_far = farthest(ref)
sfd = load_sfd(ref.nside)
print(
    f"Last-bin distance [pc]: median {np.nanmedian(dist_far):.0f}, "
    f"min {np.nanmin(dist_far):.0f}, max {np.nanmax(dist_far):.0f}"
)

if sfd is None:
    print(f"No 2D SFD map with NSIDE={ref.nside} found in {DUST2D_DIR}")
else:
    l_pix, b_pix = pixel_lb(ref.nside)
    good = np.isfinite(ebv_far) & (ebv_far > 0) & (sfd > 0)
    ratio = np.full(ref.npix, np.nan)
    ratio[good] = ebv_far[good] / sfd[good]

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
    hb = axes[0].hexbin(
        sfd[good], ebv_far[good], xscale="log", yscale="log", bins="log", gridsize=120, mincnt=1
    )
    lim = [sfd[good].min(), sfd[good].max()]
    axes[0].plot(lim, lim, "r-", lw=0.8)
    axes[0].set(xlabel="SFD 2D [mag]", ylabel="3D map, last distance bin [mag]")
    fig.colorbar(hb, ax=axes[0], label="pixels")
    axes[1].hist(np.log10(ratio[good]), bins=np.linspace(-1.5, 1, 150))
    axes[1].set(xlabel="log10(3D far end / SFD)", ylabel="number of pixels")
    plt.tight_layout()
    plt.show()

    hp.mollview(
        np.log10(ratio),
        coord=["C", "G"],
        min=-0.5,
        max=0.5,
        cmap="RdBu_r",
        unit="log10(3D far end / SFD)",
        title=f"3D map ({REF}), last bin, relative to SFD",
    )
    hp.graticule()
    plt.show()

    for label, sel in (
        ("|b| < 10 deg ", np.abs(b_pix) < 10),
        ("10 <= |b| < 30", (np.abs(b_pix) >= 10) & (np.abs(b_pix) < 30)),
        ("|b| >= 30 deg", np.abs(b_pix) >= 30),
    ):
        r = ratio[good & sel]
        p16, p50, p84 = np.percentile(r, [16, 50, 84])
        print(
            f"{label}: ratio 3D/SFD median = {p50:.3f}, 68% interval = [{p16:.3f}, {p84:.3f}]  ({r.size} pixels)"
        )

## 6. Distance reached at a given `m - M0`

As in `DustMap3D`: the distance at which distance modulus plus extinction, `5 log10(d / 10 pc) + A_X(d)`,
reaches `D_MAG` in filter `FILTER`, with `A_X = R_X * E(B-V)(d)`. Without dust this distance is
`10 pc * 10**(D_MAG / 5)`. Pixels where no distance bin gets within 0.5 mag of `D_MAG` are left blank.

In [ ]:
FILTER, D_MAG = "r", 15.2  # same defaults as rubin_sim.maf.maps.DustMap3D

try:
    from rubin_sim.phot_utils import DustValues

    R_X = dict(DustValues().r_x)
    r_x_source = "rubin_sim.phot_utils.DustValues"
except Exception:
    R_X = {
        "u": 4.145,
        "g": 3.237,
        "r": 2.273,
        "i": 1.684,
        "z": 1.323,
        "y": 1.088,
    }  # A_X / E(B-V), approximate
    r_x_source = "built-in approximate values (rubin_sim not available)"
print(f"R_{FILTER} = A_{FILTER} / E(B-V) = {R_X[FILTER]:.3f}   [{r_x_source}]")

d_free = 10.0 * 10 ** (D_MAG / 5.0)
dist_mag = distance_at_mag(ref, D_MAG, R_X[FILTER])
print(f"Distance without dust : {d_free / 1e3:.2f} kpc")
print(
    f"With dust             : median {np.nanmedian(dist_mag) / 1e3:.2f} kpc, "
    f"min {np.nanmin(dist_mag) / 1e3:.3f} kpc; undefined for {np.isnan(dist_mag).mean():.1%} of the pixels"
)

hp.mollview(
    dist_mag / 1e3,
    coord=["C", "G"],
    norm="log",
    unit="distance [kpc]",
    title=f"Distance at which m - M0 = {D_MAG} in {FILTER} ({REF})",
)
hp.graticule()
plt.show()

## 7. Comparison of the map variants

E(B-V) at 3 kpc (the `DustMap3D` default) for each NSIDE=64 variant; the other maps are shown as the difference
with the reference map.

In [ ]:
D_COMPARE = 3000
maps_cmp = {name: at_distance(cubes[name], D_COMPARE)[0] for name in variants64}
others = [n for n in variants64 if n != REF]

fig = plt.figure(figsize=(14, 4.5 * ((len(others) + 2) // 2)))
nrow = (len(others) + 2) // 2
m_ref = np.where(maps_cmp[REF] > 0, maps_cmp[REF], np.nan)
hp.mollview(
    m_ref,
    coord=["C", "G"],
    norm="log",
    min=0.01,
    max=1,
    unit="E(B-V) [mag]",
    title=f"{REF}: E(B-V) at {D_COMPARE} pc",
    sub=(nrow, 2, 1),
    fig=fig.number,
)
for i, name in enumerate(others, start=2):
    diff = maps_cmp[name] - maps_cmp[REF]
    lim = np.nanpercentile(np.abs(diff), 99)
    hp.mollview(
        diff,
        coord=["C", "G"],
        min=-lim,
        max=lim,
        cmap="RdBu_r",
        unit="delta E(B-V) [mag]",
        title=f"{name} - {REF}",
        sub=(nrow, 2, i),
        fig=fig.number,
    )
plt.show()

print(f"Ratio to {REF} at {D_COMPARE} pc (pixels with E(B-V) > 0.01 in both maps):")
for name in others:
    ok = (maps_cmp[name] > 0.01) & (maps_cmp[REF] > 0.01)
    r = maps_cmp[name][ok] / maps_cmp[REF][ok]
    p16, p50, p84 = np.percentile(r, [16, 50, 84])
    print(
        f"  {name:9s}: median = {p50:.3f}, 68% interval = [{p16:.3f}, {p84:.3f}], "
        f"|ratio - 1| > 10% for {np.mean(np.abs(r - 1) > 0.1):.1%} of the pixels"
    )

## 8. Effect of the HEALPix resolution (NSIDE 64 vs 128)

The NSIDE=128 map degraded to NSIDE=64 is compared with the native NSIDE=64 map (E(B-V) at the same distance), and
a Galactic field is shown at both resolutions.

In [ ]:
hi_name = next((n for n, c in cubes.items() if c.nside == 2 * ref.nside), None)

if RUN_NSIDE128 and hi_name is not None:
    hi = cubes[hi_name]
    t0 = time.time()
    m_hi, _ = at_distance(hi, D_COMPARE)
    print(f"{hi_name}: slice computed in {time.time() - t0:.1f} s")
    m_lo = maps_cmp[REF]
    m_hi_deg = hp.ud_grade(m_hi, ref.nside, order_in="RING", order_out="RING")
    ok = (m_lo > 0.01) & (m_hi_deg > 0.01)
    p16, p50, p84 = np.percentile(m_lo[ok] / m_hi_deg[ok], [16, 50, 84])
    print(
        f"native NSIDE={ref.nside} / NSIDE={hi.nside} degraded: median = {p50:.3f}, 68% interval = [{p16:.3f}, {p84:.3f}]"
    )

    fig = plt.figure(figsize=(13, 5.5))
    for i, (m, nside) in enumerate(((m_lo, ref.nside), (m_hi, hi.nside)), start=1):
        hp.gnomview(
            np.where(m > 0, m, np.nan),
            coord=["C", "G"],
            rot=(0, -5),
            xsize=500,
            reso=4.0,
            norm="log",
            min=0.01,
            max=1,
            unit="E(B-V) [mag]",
            title=f"NSIDE = {nside}, {D_COMPARE} pc",
            sub=(1, 2, i),
            fig=fig.number,
        )
        hp.graticule()
    plt.show()
else:
    print("NSIDE=128 comparison skipped (RUN_NSIDE128 is False or no NSIDE=128 file found).")

## Notes

* In `rubin_sim`, these maps are read with `rubin_sim.maf.maps.DustMap3D(nside=64, map_file='merged_ebv3d_nside64_defaults.fits')`;
  the slice points then receive `ebv3d_dists` and `ebv3d_ebvs` (the two arrays explored above) plus derived quantities such as
  the E(B-V) at a given distance and the distance at a given `m - M0`.
* The `bridge`, `noL19` and `merged` variants differ in which underlying 3D maps are combined and how they are joined
  (see the `BRIDGL19`, `BRIDGWID`, `DMAXL19`, `PLANCKOK` keywords in the header table above, and the source repository
  quoted in the `README`).
* Cleanup: `for c in cubes.values(): c.close()` releases the memory-mapped files.